# HOL Snowpark Demo — Python Notebook

End-to-end data pipeline using **Snowpark Python** in a Snowflake Notebook.

**Layers**: `RAW` → `CURATED` → `CONSUMPTION`

**Features demonstrated**:
- Snowpark Session & DataFrame API
- Semi-structured (VARIANT) data processing
- FLATTEN / lateral joins in Python
- Window functions & aggregations
- Writing results back to Snowflake tables
- Querying Dynamic Tables & Materialized Views
- Visualization with matplotlib

In [ ]:
# Cell 1: Setup — Connect to Snowflake via Snowpark
from snowflake.snowpark import Session
from snowflake.snowpark.functions import (
    col, lit, sum as sum_, count, avg as avg_, max as max_, min as min_,
    when, datediff, current_timestamp, to_date, date_trunc,
    parse_json, flatten, iff, row_number, rank, ntile,
    array_agg, object_construct
)
from snowflake.snowpark import Window
import json

# For Snowflake Notebooks use get_active_session()
# For local dev, use Session.builder.configs({...}).create()
session = Session.builder.getOrCreate()  # or get_active_session()

session.sql('USE ROLE ACCOUNTADMIN').collect()
session.sql('USE WAREHOUSE HOL_WH').collect()
session.sql('USE DATABASE HOL_DB').collect()

print(f'Connected: {session.get_current_account()}')
print(f'Warehouse: {session.get_current_warehouse()}')
print(f'Database:  {session.get_current_database()}')

## 1. Explore RAW Layer
Read the raw tables as Snowpark DataFrames and inspect their schemas.

In [ ]:
# Cell 2: Read RAW tables
customers_df = session.table('RAW.CUSTOMERS')
products_df  = session.table('RAW.PRODUCTS')
orders_df    = session.table('RAW.ORDERS')
events_df    = session.table('RAW.WEBSITE_EVENTS')

print('=== RAW.CUSTOMERS ===')
customers_df.show(5)
print(f'Row count: {customers_df.count()}')

print('\n=== RAW.PRODUCTS ===')
products_df.show(5)
print(f'Row count: {products_df.count()}')

print('\n=== RAW.ORDERS ===')
print('Schema:')
for field in orders_df.schema.fields:
    print(f'  {field.name}: {field.datatype}')
print(f'Row count: {orders_df.count()}')

print('\n=== RAW.WEBSITE_EVENTS ===')
print(f'Row count: {events_df.count()}')
events_df.group_by('EVENT_TYPE').count().show()

## 2. Process Semi-Structured Data (VARIANT)
Extract nested JSON fields from the `ORDER_DETAILS` VARIANT column.

In [ ]:
# Cell 3: Flatten VARIANT — extract line items from ORDER_DETAILS

# Method 1: Using Snowpark col[] accessor for nested paths
order_shipping = orders_df.select(
    col('ORDER_ID'),
    col('STATUS'),
    col('ORDER_DETAILS')['shipping']['method'].cast('VARCHAR').alias('SHIP_METHOD'),
    col('ORDER_DETAILS')['shipping']['cost'].cast('FLOAT').alias('SHIP_COST'),
    col('ORDER_DETAILS')['shipping']['address']['city'].cast('VARCHAR').alias('SHIP_CITY'),
    col('ORDER_DETAILS')['payment']['method'].cast('VARCHAR').alias('PAY_METHOD')
)
print('=== Shipping & Payment Info (extracted from VARIANT) ===')
order_shipping.show()

# Method 2: Using SQL FLATTEN via session.sql for line items
line_items = session.sql("""
    SELECT
        o.ORDER_ID,
        li.VALUE:product_name::VARCHAR   AS PRODUCT,
        li.VALUE:quantity::INT           AS QTY,
        li.VALUE:unit_price::FLOAT       AS PRICE,
        li.VALUE:discount::FLOAT         AS DISCOUNT
    FROM RAW.ORDERS o,
    LATERAL FLATTEN(INPUT => o.ORDER_DETAILS:line_items) li
""")
print('\n=== Flattened Line Items ===')
line_items.show(10)
print(f'Total line items across all orders: {line_items.count()}')

In [ ]:
# Cell 4: Parse website events VARIANT data
events_parsed = events_df.select(
    col('EVENT_ID'),
    col('EVENT_TIME'),
    col('CUSTOMER_ID'),
    col('EVENT_TYPE'),
    col('EVENT_DATA')['page'].cast('VARCHAR').alias('PAGE_URL'),
    col('EVENT_DATA')['referrer'].cast('VARCHAR').alias('REFERRER'),
    col('EVENT_DATA')['session_id'].cast('VARCHAR').alias('SESSION_ID'),
    col('EVENT_DATA')['device']['type'].cast('VARCHAR').alias('DEVICE_TYPE'),
    col('EVENT_DATA')['device']['browser'].cast('VARCHAR').alias('BROWSER'),
    col('EVENT_DATA')['device']['os'].cast('VARCHAR').alias('OS'),
    col('EVENT_DATA')['duration_sec'].cast('INT').alias('DURATION_SEC'),
    col('EVENT_DATA')['product_name'].cast('VARCHAR').alias('PRODUCT_NAME'),
    col('EVENT_DATA')['cart_total'].cast('FLOAT').alias('CART_TOTAL'),
    col('EVENT_DATA')['coupon_applied'].cast('VARCHAR').alias('COUPON')
)

print('=== Parsed Website Events ===')
events_parsed.show(10)

# Device breakdown
print('\n=== Events by Device ===')
events_parsed.group_by('DEVICE_TYPE').agg(count('*').alias('EVENT_COUNT')).show()

## 3. Transform RAW → CURATED
Join tables, apply business logic, and write enriched data to the CURATED schema.

In [ ]:
# Cell 5: Build enriched orders with customer info + extracted VARIANT fields

# Prepare customer dimension
cust_dim = customers_df.select(
    col('CUSTOMER_ID'),
    (col('FIRST_NAME') + lit(' ') + col('LAST_NAME')).alias('CUSTOMER_NAME'),
    col('CITY').alias('CUSTOMER_CITY'),
    col('COUNTRY').alias('CUSTOMER_COUNTRY')
)

# Join and enrich
enriched_orders = orders_df.join(cust_dim, 'CUSTOMER_ID').select(
    col('ORDER_ID'),
    col('CUSTOMER_ID'),
    col('CUSTOMER_NAME'),
    col('CUSTOMER_CITY'),
    col('CUSTOMER_COUNTRY'),
    col('ORDER_DATE'),
    col('STATUS'),
    col('TOTAL_AMOUNT'),
    col('ORDER_DETAILS')['shipping']['method'].cast('VARCHAR').alias('SHIPPING_METHOD'),
    col('ORDER_DETAILS')['shipping']['cost'].cast('FLOAT').alias('SHIPPING_COST'),
    col('ORDER_DETAILS')['payment']['method'].cast('VARCHAR').alias('PAYMENT_METHOD'),
    # Categorize order size
    when(col('TOTAL_AMOUNT') >= 800, lit('PREMIUM'))
    .when(col('TOTAL_AMOUNT') >= 400, lit('LARGE'))
    .when(col('TOTAL_AMOUNT') >= 150, lit('MEDIUM'))
    .otherwise(lit('SMALL')).alias('ORDER_SIZE')
)

# Add window functions: running total per customer
window_spec = Window.partition_by('CUSTOMER_ID').order_by('ORDER_DATE')

enriched_with_windows = enriched_orders.with_column(
    'ORDER_SEQUENCE', row_number().over(window_spec)
).with_column(
    'RUNNING_TOTAL', sum_(col('TOTAL_AMOUNT')).over(
        Window.partition_by('CUSTOMER_ID').order_by('ORDER_DATE')
        .rows_between(Window.UNBOUNDED_PRECEDING, Window.CURRENT_ROW)
    )
)

# Write to CURATED
enriched_with_windows.write.mode('overwrite').save_as_table('CURATED.ORDERS_ENRICHED_NOTEBOOK')

print('=== Enriched Orders Written to CURATED ===')
session.table('CURATED.ORDERS_ENRICHED_NOTEBOOK').show(10)
print(f'Rows written: {session.table("CURATED.ORDERS_ENRICHED_NOTEBOOK").count()}')

In [ ]:
# Cell 6: Build CONSUMPTION aggregations in Python

# Revenue by product category
line_items_df = session.sql("""
    SELECT
        li.VALUE:product_name::VARCHAR AS PRODUCT_NAME,
        li.VALUE:quantity::INT AS QTY,
        li.VALUE:unit_price::FLOAT AS PRICE,
        li.VALUE:discount::FLOAT AS DISCOUNT,
        o.ORDER_DATE,
        o.CUSTOMER_ID
    FROM RAW.ORDERS o,
    LATERAL FLATTEN(INPUT => o.ORDER_DETAILS:line_items) li
""")

# Join with products for category
product_sales = line_items_df.join(
    products_df.select('PRODUCT_NAME', 'CATEGORY', 'SUB_CATEGORY'),
    'PRODUCT_NAME'
).with_column(
    'LINE_TOTAL',
    (col('QTY') * col('PRICE')) - col('DISCOUNT')
)

# Aggregate by category
category_summary = product_sales.group_by('CATEGORY', 'SUB_CATEGORY').agg(
    sum_('LINE_TOTAL').alias('TOTAL_REVENUE'),
    sum_('QTY').alias('TOTAL_UNITS'),
    count('*').alias('LINE_ITEM_COUNT')
).sort(col('TOTAL_REVENUE').desc())

# Save to consumption
category_summary.write.mode('overwrite').save_as_table('CONSUMPTION.CATEGORY_SUMMARY_NOTEBOOK')

print('=== Category Revenue Summary ===')
category_summary.show()

## 4. Query Dynamic Tables & Materialized Views
Read from the objects created by the SQL pipeline.

In [ ]:
# Cell 7: Query Dynamic Tables and Materialized Views via Snowpark

# Customer 360 from Dynamic Table
print('=== CONSUMPTION.DT_CUSTOMER_360 (Dynamic Table) ===')
c360 = session.table('CONSUMPTION.DT_CUSTOMER_360')
c360.select(
    'CUSTOMER_ID', 'FIRST_NAME', 'LAST_NAME',
    'LIFETIME_VALUE', 'TOTAL_ORDERS', 'CUSTOMER_SEGMENT',
    'TOTAL_EVENTS', 'CART_CONVERSION_RATE'
).sort(col('LIFETIME_VALUE').desc()).show()

# Product performance from Materialized View
print('\n=== CONSUMPTION.DT_PRODUCT_PERFORMANCE (Dynamic Table) ===')
pp = session.table('CONSUMPTION.DT_PRODUCT_PERFORMANCE')
pp.select(
    'PRODUCT_NAME', 'CATEGORY', 'TOTAL_REVENUE',
    'TOTAL_UNITS_SOLD', 'UNIQUE_BUYERS'
).sort(col('TOTAL_REVENUE').desc()).show()

# Daily sales from Dynamic Table
print('\n=== CONSUMPTION.DT_DAILY_SALES (Dynamic Table) ===')
daily = session.table('CONSUMPTION.DT_DAILY_SALES')
daily.sort('SALE_DATE').show()

## 5. Visualization
Create charts from our pipeline data using matplotlib.

In [ ]:
# Cell 8: Visualizations
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# --- Chart 1: Revenue by Product Category ---
cat_data = session.table('CONSUMPTION.CATEGORY_SUMMARY_NOTEBOOK').sort(
    col('TOTAL_REVENUE').desc()
).to_pandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
ax1 = axes[0]
bars = ax1.barh(cat_data['CATEGORY'], cat_data['TOTAL_REVENUE'], color=['#29B5E8', '#71D3F7', '#FFCC33', '#FF6633'])
ax1.set_xlabel('Revenue ($)')
ax1.set_title('Revenue by Product Category')
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax1.invert_yaxis()

# --- Chart 2: Customer Segments ---
seg_data = session.table('CONSUMPTION.DT_CUSTOMER_360').group_by('CUSTOMER_SEGMENT').agg(
    count('*').alias('COUNT')
).to_pandas()

ax2 = axes[1]
colors = ['#29B5E8', '#71D3F7', '#FFCC33', '#FF6633']
ax2.pie(seg_data['COUNT'], labels=seg_data['CUSTOMER_SEGMENT'],
        autopct='%1.0f%%', colors=colors[:len(seg_data)])
ax2.set_title('Customer Segmentation')

plt.tight_layout()
plt.show()

# --- Chart 3: Daily Revenue ---
daily_data = session.table('CONSUMPTION.DT_DAILY_SALES').sort('SALE_DATE').to_pandas()

fig2, ax3 = plt.subplots(figsize=(12, 4))
ax3.bar(daily_data['SALE_DATE'].astype(str), daily_data['GROSS_REVENUE'], color='#29B5E8')
ax3.set_xlabel('Date')
ax3.set_ylabel('Revenue ($)')
ax3.set_title('Daily Gross Revenue')
ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 6. Pipeline Monitoring
Check task history, stream status, and dynamic table refresh status.

In [ ]:
# Cell 9: Monitor pipeline — tasks, streams, dynamic tables

# Stream status
print('=== Stream Status ===')
streams = session.sql("""
    SELECT
        NAME,
        STALE_AFTER,
        TABLE_NAME AS SOURCE_TABLE,
        TYPE
    FROM INFORMATION_SCHEMA.STREAMS
    WHERE TABLE_SCHEMA = 'RAW'
""")
streams.show()

# Dynamic table refresh history
print('\n=== Dynamic Table Refresh History ===')
dt_history = session.sql("""
    SELECT
        NAME,
        SCHEMA_NAME,
        STATE,
        TARGET_LAG,
        REFRESH_MODE
    FROM INFORMATION_SCHEMA.DYNAMIC_TABLES
    WHERE TABLE_CATALOG = 'HOL_DB'
    ORDER BY SCHEMA_NAME, NAME
""")
dt_history.show()

# Task history
print('\n=== Recent Task Runs ===')
task_history = session.sql("""
    SELECT
        NAME,
        STATE,
        SCHEDULED_TIME,
        COMPLETED_TIME,
        ERROR_CODE,
        ERROR_MESSAGE
    FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
        SCHEDULED_TIME_RANGE_START => DATEADD('hour', -1, CURRENT_TIMESTAMP()),
        RESULT_LIMIT => 10
    ))
    ORDER BY SCHEDULED_TIME DESC
""")
task_history.show()

# Pipeline data counts
print('\n=== Data Layer Row Counts ===')
layers = [
    ('RAW', 'CUSTOMERS'), ('RAW', 'PRODUCTS'), ('RAW', 'ORDERS'), ('RAW', 'WEBSITE_EVENTS'),
    ('CURATED', 'DT_ORDER_ENRICHED'), ('CURATED', 'DT_EVENT_PARSED'),
    ('CONSUMPTION', 'DT_CUSTOMER_360'), ('CONSUMPTION', 'DT_DAILY_SALES'),
]
for schema, table in layers:
    try:
        cnt = session.table(f'{schema}.{table}').count()
        print(f'  {schema}.{table}: {cnt} rows')
    except Exception as e:
        print(f'  {schema}.{table}: ERROR - {e}')

## Summary

This notebook demonstrated:
1. **Snowpark Session** — connecting and setting context
2. **DataFrame API** — reading tables, selecting columns, joins
3. **Semi-structured data** — extracting VARIANT fields with `col[]` and `LATERAL FLATTEN`
4. **Transformations** — window functions, aggregations, conditional logic
5. **Write-back** — saving DataFrames as Snowflake tables
6. **Dynamic Tables & MVs** — querying pipeline outputs
7. **Visualization** — matplotlib charts from Snowpark data
8. **Monitoring** — stream status, task history, DT refresh state